# ML-07: Baseline Action Score and Top-10 Review

Week 4. Before any model, a hand-written rule: the thing the week 5 model has to beat on the same data and the same metric. Same slice as week 3: pages at the 1 April 2026 decision date, features from March only, label = declined in April relative to the page's own client. June stays sealed.

Lane check: I'm confirming Lane 2 (Refresh / Content Opportunity Scoring) as my locked lane. Weeks 1 to 3 all point the same way, and nothing in the warehouse slice argued for switching.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "work" / "scripts" / "warehouse.py").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "work" / "scripts"))
import warehouse as W  # shared loader + Lane 2 frame builder (weeks 3-6)

con = W.connect()

frame = W.build_frame(con, feature_month="2026-03", label_month="2026-04")
y = frame["label"]
print(f"pages: {len(frame):,}   clients: {frame['client_hash_id'].nunique()}   base rate: {y.mean():.1%}")

pages: 84,347   clients: 36   base rate: 27.0%


## 1. My rule and its reason codes

The rule idea I started with, in plain words: *big pages that are under-capturing clicks for their position are the ones to refresh first.* It leans on two signals, and both sit behind real FlyRank flags from this week's session: search volume (behind quick-win) and CTR against what the position normally earns (behind the CTR-fix logic). Each one gets checked against the April outcome before it's allowed into a rule.

Position tiers follow the paper: top 3, page 1 (4 to 10), striking distance (11 to 20), pages 3 to 5 (21 to 50), deep (50+). CTR gap = the page's March CTR minus the median March CTR of its tier.

In [2]:
frame["position_tier"] = pd.cut(
    frame["avg_position"], [0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"], include_lowest=True,
)
frame["tier_median_ctr"] = frame.groupby("position_tier", observed=True)["ctr_pct"].transform("median")
frame["ctr_gap"] = frame["ctr_pct"] - frame["tier_median_ctr"]

def bucket_table(bucket):
    t = frame.groupby(bucket, observed=True)["label"].agg(n="size", decline_rate="mean")
    t["decline_rate"] = t["decline_rate"].map("{:.1%}".format)
    return t

# Signal A: volume (quick-win). Do bigger pages decline more often?
frame["march_impressions"] = pd.cut(frame["imp"], [100, 300, 1_000, 3_000, 10_000, np.inf], right=False,
                                    labels=["100-300", "300-1K", "1K-3K", "3K-10K", "10K+"])
bucket_table("march_impressions")

,n,decline_rate
march_impressions,,
100-300,22638,31.3%
300-1K,23539,29.9%
1K-3K,18510,25.6%
3K-10K,14276,20.2%
10K+,5384,19.2%


**Volume: OPPOSITE.** I expected the biggest pages to be the most at risk, since they have the most to lose. The data says the reverse: the decline rate falls steadily as March impressions rise, from 31.3% in the smallest bucket to 19.2% in the 10K+ bucket, with thousands of pages in every bucket. Large pages are steadier. That saved the rule: if volume had gone into the score as a risk multiplier, it would have pushed the queue toward the pages least likely to decline. Volume still matters for how much a decline costs, so it comes back in week 5 as a demand weight, not as risk.

In [3]:
# Signal B: CTR against its position tier (CTR-fix logic). Do under-clicked pages decline more often?
frame["ctr_vs_tier"] = pd.qcut(frame["ctr_gap"].rank(method="first"), 5,
                               labels=["far below", "below", "near", "above", "far above"])
t = bucket_table("ctr_vs_tier")
t["median_gap_pts"] = frame.groupby("ctr_vs_tier", observed=True)["ctr_gap"].median().round(3)
t

,n,decline_rate,median_gap_pts
ctr_vs_tier,,,
far below,16870,39.5%,-0.180
below,16869,30.5%,-0.077
near,16869,27.9%,0.000
above,16869,20.5%,0.154
far above,16870,16.4%,0.526


**CTR vs position: CONFIRMED.** Pages clicked far less than their tier's typical CTR declined at 39.5% in April; pages clicked far more than typical declined at 16.4%. The gradient is steady across all five buckets of about 16,900 pages each. A page that's shown but not chosen looks like a page about to lose its spot.

Volume failing left the rule without its "which pages" signal, so I checked one replacement before using it: late-March momentum (per-day impressions in the second half of March vs the first half, both before the decision date).

In [4]:
# Signal C (replacement for volume): late-March momentum.
frame["momentum_bucket"] = pd.qcut(frame["momentum"], 5,
                                   labels=["sliding hard", "sliding", "flat", "rising", "rising hard"])
bucket_table("momentum_bucket")

,n,decline_rate
momentum_bucket,,
sliding hard,16870,51.9%
sliding,16870,28.0%
flat,16868,20.0%
rising,16869,17.4%
rising hard,16870,17.7%


**Momentum: CONFIRMED.** Pages that were already sliding hard in the second half of March declined at 51.9% in April, against about 17 to 18% for pages that were rising. It's the strongest of the three.

**The rule, in three sentences.** A page is flagged when its impressions were sliding through late March (momentum below -0.1, roughly a 10% drop between the two halves of the month) *and* it was clicked less than the median page in its position tier. Flagged pages are ranked by how hard they were sliding. Everything else scores zero.

One reason code, one action:

- `sliding_and_under_clicked`: impressions fell through late March and CTR sits below its position tier's median.
- Action label `refresh_and_rewrite_snippet`: refresh the content and rewrite the title and meta description.

## 2. Build the ranked queue (writes the CSV)

In [5]:
SLIDE_CUT = -0.1

flagged = frame["momentum"].lt(SLIDE_CUT) & frame["ctr_gap"].lt(0)
frame["baseline_score"] = np.where(flagged, -frame["momentum"], 0.0)
frame["reason_code"] = np.where(flagged, "sliding_and_under_clicked", "")
frame["action"] = np.where(flagged, "refresh_and_rewrite_snippet", "no_action")

queue = (frame.sort_values(["baseline_score", "imp"], ascending=[False, False])
              .reset_index(drop=True))
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

out_dir = ROOT / "work" / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
export_cols = ["rank", "content_hash_id", "client_hash_id", "baseline_score", "reason_code", "action",
               "imp", "momentum", "avg_position", "position_tier", "ctr_pct", "tier_median_ctr"]
queue[export_cols].to_csv(out_dir / "baseline_action_score.csv", index=False)

print(f"flagged pages: {flagged.sum():,} of {len(frame):,} ({flagged.mean():.1%})")
print(f"wrote work/outputs/baseline_action_score.csv with {len(queue):,} rows")

flagged pages: 14,399 of 84,347 (17.1%)
wrote work/outputs/baseline_action_score.csv with 84,347 rows


In [6]:
import json
from sklearn.metrics import average_precision_score

lost = frame["lost_per_day"]
score = frame["baseline_score"]
rng = np.random.default_rng(42)
random_score = rng.random(len(frame))

metrics = {
    "decision_date": "2026-04-01",
    "rows": int(len(frame)),
    "base_rate": round(float(y.mean()), 4),
    "flagged": int(flagged.sum()),
    "precision_of_flagged_set": round(float(y[flagged].mean()), 4),
    "baseline_precision_at_20": round(W.precision_at_k(score, y, 20), 4),
    "baseline_precision_at_50": round(W.precision_at_k(score, y, 50), 4),
    "baseline_demand_captured_at_50": round(W.demand_captured_at_k(score, y, lost, 50), 4),
    "baseline_precision_at_500": round(W.precision_at_k(score, y, 500), 4),
    "baseline_precision_at_1000": round(W.precision_at_k(score, y, 1000), 4),
    "baseline_average_precision": round(float(average_precision_score(y, score)), 4),
    "random_precision_at_50": round(W.precision_at_k(random_score, y, 50), 4),
    "random_demand_captured_at_50": round(W.demand_captured_at_k(random_score, y, lost, 50), 4),
}
(out_dir / "w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
pd.Series(metrics).to_frame("value")

,value
decision_date,2026-04-01
rows,84347
base_rate,0.2699
flagged,14399
precision_of_flagged_set,0.4758
baseline_precision_at_20,1.0
baseline_precision_at_50,0.96
baseline_demand_captured_at_50,0.0691
baseline_precision_at_500,0.874
baseline_precision_at_1000,0.832


The rule flags 14,399 pages (17.1% of the slice), and 47.6% of the flagged set declined against a 27.0% base rate. At the very top it's close to perfect: 20 of the first 20 and 48 of the first 50 declined, where a random 50 got 12. Further down it still holds, at 0.87 precision in the top 500 and 0.83 in the top 1,000.

The weak number is demand captured@50: 6.9%. The queue finds pages that declined, but mostly small ones, so it recovers little of the April impressions that were really lost. Average precision is 0.39, held down by the 83% of pages the rule gives a flat zero. Those are the two places a model has room in week 5. Section 4 shows why the top 50 looks so good, and it isn't entirely good news.

## 3. Top-10 review

The ten pages the rule puts first, with what actually happened in April next to them. The April numbers are for review only; the rule never saw them.

In [7]:
top10 = queue.head(10).copy()
top10["april_change_pct"] = top10["change_pct"].round(1)
top10["client_median_change"] = top10["client_median_change"].round(1)
top10["client_pages_in_top50"] = top10["client_hash_id"].map(queue.head(50)["client_hash_id"].value_counts())
top10[["rank", "content_hash_id", "baseline_score", "imp", "days_visible", "avg_position", "ctr_pct",
       "tier_median_ctr", "content_age_days", "client_pages_in_top50", "april_change_pct",
       "client_median_change", "label"]].round(3)

,rank,content_hash_id,baseline_score,imp,days_visible,avg_position,ctr_pct,tier_median_ctr,content_age_days,client_pages_in_top50,april_change_pct,client_median_change,label
0,1,content_9c057b66c30a3abb,7.044,83834.0,31,0.116,0.001,0.202,244,4,-99.8,-10.8,1
1,2,content_fe8328ab251759cc,5.743,5250.0,11,9.286,0.000,0.180,36,4,-100.0,25.9,1
2,3,content_b732c6553189c45a,4.962,2129.0,13,6.913,0.000,0.180,159,2,-100.0,37.2,1
3,4,content_f97d175377d97f04,4.854,6639.0,20,10.951,0.045,0.092,156,6,-100.0,-15.1,1
4,5,content_7f739375dd053693,4.586,1549.0,13,7.720,0.000,0.180,315,13,-99.5,-45.8,1
5,6,content_31c5233585533e77,4.468,3528.0,23,0.775,0.113,0.202,187,4,-100.0,-10.8,1
6,7,content_32c750b821d322d2,4.434,1409.0,16,12.504,0.000,0.092,217,13,-100.0,-45.8,1
7,8,content_dfbc52ce403be0d9,4.336,1131.0,15,8.319,0.000,0.180,376,5,-100.0,-53.4,1
8,9,content_6e8d8b092a2d9431,4.164,3462.0,30,6.340,0.058,0.180,148,13,-90.7,-45.8,1
9,10,content_c68f1a60224aee0a,4.095,886.0,10,7.376,0.000,0.180,60,5,-100.0,-53.4,1


All ten declined by the label, so the review is about whether the *action* is right and what could still make each pick a waste of a reviewer's time.

1. **Refresh + snippet rewrite.** Biggest page in the top 10 (83.8K March impressions), and it slid hardest. Wrong if the average position of 0.12 (impossible in Search Console, which starts at 1) points to a tracking problem on this URL rather than a content one; its tier, and so its CTR gap, would then be wrong too.
2. **Refresh + snippet rewrite.** A 36-day-old page that was visible on only 11 March days and went to zero in April while its client grew 26%. Wrong if it was unpublished or merged on purpose; a young page vanishing that fast looks more like a decision than decay.
3. **Refresh + snippet rewrite.** Page-1 position, zero clicks, visible 13 days, then zero in April against a client up 37%. Wrong if it was deindexed (noindex, 404, redirect); no content edit fixes that, so the right action is an indexing check.
4. **Refresh + snippet rewrite.** Striking-distance page with half its tier's CTR, gone in April. Wrong if its client's six top-50 pages share one cause, such as a section of the site moving; that's one fix, not six reviews.
5. **Refresh + snippet rewrite.** From the client that holds 13 of the top 50 slots, whose median page fell 46% in April. Wrong if the drop is site-wide; the client adjustment still flags it, but a site-level check should come before page edits.
6. **Refresh + snippet rewrite.** Average position 0.78, the same impossible-position quirk as rank 1, and the same client. Wrong for the same reason: the CTR gap is measured against the top-3 tier only because of a suspect position value.
7. **Refresh + snippet rewrite.** Same 13-page client as rank 5, zero clicks at position 12.5. Wrong if it's part of the same site-wide event; reviewing it separately duplicates work.
8. **Refresh + snippet rewrite.** About a year old, page 1, zero clicks, gone in April. Wrong in a cost sense even if right: about 36 impressions a day is the least demand in the top 10, so a reviewer's hour buys little.
9. **Refresh + snippet rewrite.** The pick that best fits the rule as intended: visible on 30 of 31 days, page 1, a third of its tier's CTR, then down 91%. Wrong if its client's 46% median drop explains most of it, which the label's 20-point margin only partly controls for.
10. **Refresh + snippet rewrite.** A 60-day-old page with 886 March impressions, visible 10 days, gone in April. Wrong if it was a short-lived page removed on purpose, and it's the smallest page here.

## 4. Weak picks + leakage check

In [8]:
top50 = queue.head(50)
print("top-50 pages per client:")
print(top50["client_hash_id"].value_counts().to_string())
print()
print(f"top-50 pages visible on fewer than 20 of 31 March days: {(top50['days_visible'] < 20).sum()}")
print(f"top-50 pages with under 300 March impressions: {(top50['imp'] < 300).sum()}")
print(f"top-50 pages with zero impressions in April (review only): {(top50['imp_next'] == 0).sum()}")

status = con.sql(f"SELECT content_hash_id, is_deleted FROM read_parquet('{W.file_path('dim_content.parquet')}')").df()
print(f"top-50 pages marked deleted in the July snapshot: {int(top50.merge(status, on='content_hash_id')['is_deleted'].sum())}")
print(f"pages in the frame with average position below 1: {(frame['avg_position'] < 1).sum()} ({(frame['avg_position'] < 1).mean():.1%})")
print()
for k in [50, 500, 1000, 2000]:
    head = queue.head(k)
    print(f"top {k:>4}: precision {head['label'].mean():.3f}   zero April impressions {(head['imp_next'] == 0).mean():.1%}")

top-50 pages per client:
client_hash_id
client_62f4a7e64f5e0096    13
client_20259bd6705d81d4     6
client_65de48885f4ef01b     5
client_400c21c81c8b46ef     5
client_73cda7b4e4f265ea     4
client_0fa64a184f18a4a0     4
client_cd12bcfd98942aa1     2
client_c182d11e4862a37d     2
client_a80fca3f171ed1de     2
client_3f0ce4d44fe94f3d     2
client_2094c6eb080311d5     2
client_e5c2aa26a8598242     1
client_fef1a8f436438636     1
client_157ffe4d4a595515     1

top-50 pages visible on fewer than 20 of 31 March days: 33
top-50 pages with under 300 March impressions: 0
top-50 pages with zero impressions in April (review only): 23
top-50 pages marked deleted in the July snapshot: 2
pages in the frame with average position below 1: 485 (0.6%)

top   50: precision 0.960   zero April impressions 46.0%
top  500: precision 0.874   zero April impressions 20.4%
top 1000: precision 0.832   zero April impressions 12.1%
top 2000: precision 0.779   zero April impressions 7.1%


In [9]:
# Leakage check: every input to the rule is computed from March or fixed metadata.
rule_inputs = {"momentum": "March days 1-15 vs 16-31",
               "ctr_pct": "March clicks / March impressions",
               "avg_position": "March, impression-weighted",
               "tier_median_ctr": "median March CTR of the tier, same March frame",
               "imp": "March impressions (tie-breaker only)"}
label_side = {"imp_next", "change_pct", "client_median_change", "vs_client", "label", "lost_per_day"}
product_flags = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"}
assert not set(rule_inputs) & label_side, "label-side column used by the rule"
assert not set(frame.columns) & product_flags, "a product flag is present in the frame"
pd.Series(rule_inputs, name="window").to_frame()

,window
momentum,March days 1-15 vs 16-31
ctr_pct,March clicks / March impressions
avg_position,"March, impression-weighted"
tier_median_ctr,"median March CTR of the tier, same March frame"
imp,March impressions (tie-breaker only)


What section 4 shows:

- **The top is "vanished pages", not "slipping pages".** 23 of the top 50 had zero impressions in April and 33 were visible on fewer than 20 March days. The slide score rewards pages that dropped out of search during March, and those almost always stay out. They're real declines, but the action label is wrong for them. A page that disappears needs an indexing and status check (noindex, 404, redirect, merge), not a snippet rewrite. Two of the top 50 are even marked deleted in the July snapshot. By the top 1,000 the zero-April share falls to about 12% and the picks look like the slipping pages the rule was written for.
- **One client takes 13 of the 50 slots.** A team working one account at a time would see a very different queue, and a site-wide event at that client could explain most of its picks.
- **A data quirk.** 485 pages (0.6%) have an average position below 1, which Search Console can't produce. Two of them are in my top 10. They should be excluded or fixed upstream before the tier-based CTR gap is trusted.
- **Leakage check passes.** Every rule input comes from March or from the March frame itself; no April column, no label-derived column and no product flag is in the frame.

For week 5 this sets a hard target at 50 (0.96 is close to the ceiling) and a soft one everywhere else: demand captured, average precision, and the pages below the flagged set. A model that only matches the rule at the top but finds bigger declines is still the better tool. Next iteration of the rule itself: a second action, `check_indexing`, for pages visible on fewer than about half the month's days.

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors and writes `work/outputs/baseline_action_score.csv`
- [x] No client names, URLs, or private queries anywhere (pseudonymized hashes and aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, with the metrics JSON in `work/outputs/`